# Memory

In [2]:
# lanchain.memory
# 대부분 0.3.1 부터 deprecated 됨. LangGraph 로의 사용을 권하고 있슴.


# from langchain.memory.buffer import ConversationBufferMemory
# from langchain.memory.buffer_window import ConversationBufferWindowMemory
# from langchain.memory.summary import ConversationSummaryMemory
# from langchain.memory.summary_buffer import ConversationSummaryBufferMemory


In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.messages.human import HumanMessage
from langchain_core.messages.ai import AIMessage


In [6]:
llm = ChatOpenAI(temperature=0.1)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful AI talking to a human'),
    ('human', '{question}')
])

chain = prompt | llm

def invoke_chain(question):
    result = chain.invoke({'question' : question})
    print('🐻‍❄️', result.content)

In [7]:
invoke_chain('My name is Milo')

🐻‍❄️ Hello Milo! How can I assist you today?


In [8]:
invoke_chain('Guess what my name is')

🐻‍❄️ I'm not able to guess your name, but I'd be happy to address you by whatever name you'd like to share with me. What should I call you?


In [ ]:
# AI 는 용청한 내용에 대한 상태정보를 기억하지 않는다.

# 챗봇은 대화의 상태를 기억해야 한다 -> 문맥에 맞는 대화를 하려면?
 
# 이전 대화의 내용을 기억하여 AI 에게 요청해야 한다

# Langchain 에서는 이를 Memory 객체들 제공하여 지원했었음
#         새로운 버전부터는 대부분의 Memory 객체들 삭제됨,,,,, 그래서 LangGraph를 사용하여 정보 저장/유지

# 메모리 구현 (수동) - MessagesPlaceHolder 사용

In [14]:
from langchain_core.prompts.chat import MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

chat_history = []

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful AI talking to a human'),

    # 이전의 chat_history 내역이 들어와야 한다. 어떻게?
    # 'history' 라는 키값으로 전달해주면 된다.
    MessagesPlaceholder(variable_name='history'),
    
    ('human', '{question}')
])

chain = prompt | llm

def invoke_chain(question, history):
    result = chain.invoke({'question' : question, 'history':chat_history})
    print('🐻‍❄️', result.content)
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))

print('🤖 chat_history : ', chat_history)
invoke_chain('My name is Milo', chat_history)
print('🤖 chat_history : ', chat_history)
invoke_chain('Guess what my name is', chat_history)
print('🤖 chat_history : ', chat_history)

🤖 chat_history :  []
🐻‍❄️ Hello Milo! How can I assist you today?
🤖 chat_history :  [HumanMessage(content='My name is Milo', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Milo! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
🐻‍❄️ Based on the information you provided earlier, I would guess that your name is Milo. Am I correct?
🤖 chat_history :  [HumanMessage(content='My name is Milo', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Milo! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Guess what my name is', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the information you provided earlier, I would guess that your name is Milo. Am I correct?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


# Conversation KGMemory

In [16]:
# 대화중에 '엔티티'의 knowledge graph 를 형성한다 => 가장 중요한 것들만 추출한 요약본.
# knowledge graph 는 history 를 가지고 오지 않는다.  대신 '엔티티' 를 가지고 옴

In [15]:
from langchain_community.memory.kg import ConversationKGMemory

In [18]:
memory=ConversationKGMemory(
    llm=llm, # 이 또한 LLM 을 사용하는 Memory 다 -> Knowledge Graph 를 만든다. (가장 중요한것만 뽑아낸 요약본)
    return_messages=True, # history 에 AIMessage 와 HumanMessage로 저장
)

In [20]:
memory.save_context(
    {'input':'Hi I am John'},
    {'output':'Hello John'}
)

In [23]:
memory.load_memory_variables({'input' : 'WHo is John?'})

{'history': []}

In [24]:
memory.save_context(
    {'input':'I live in South Korea'},
    {'output':'Wow that is so cool!'},
)

In [25]:
memory.load_memory_variables({'input' : 'WHo is John?'})

{'history': []}

# RunnablePassThrough

In [28]:
from langchain_core.runnables.passthrough import RunnablePassthrough

llm = ChatOpenAI(temperature=0.1)

chat_history = []

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful AI talking to a human'),

    # 이전의 chat_history 내역이 들어와야 한다. 어떻게?
    # 'history' 라는 키값으로 전달해주면 된다.
    MessagesPlaceholder(variable_name='history'),
    
    ('human', '{question}')
])

# chain 호출할때마다 과거 대화내역이 자동으로 llm에 전달되게 하고 싶다면

def load_memory(_):
    # print('🐱', xxx)
    return chat_history

# RunnablePassThrough는 전달받은 key를 체인의 다음요소까지 전달(key 값 변경 없이)
chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

def invoke_chain(question, history):
    result = chain.invoke({'question' : question}) # invoke 호출코드에서 chat history를 매번 입력하는건 불편
    print('🐻‍❄️', result.content)
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))

print('🤖 chat_history : ', chat_history)
invoke_chain('My name is Milo', chat_history)
print('🤖 chat_history : ', chat_history)
invoke_chain('Guess what my name is', chat_history)
print('🤖 chat_history : ', chat_history)

🤖 chat_history :  []
🐻‍❄️ Hello Milo! How can I assist you today?
🤖 chat_history :  [HumanMessage(content='My name is Milo', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Milo! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
🐻‍❄️ Based on the information you provided earlier, I would guess that your name is Milo. Am I correct?
🤖 chat_history :  [HumanMessage(content='My name is Milo', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Milo! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Guess what my name is', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the information you provided earlier, I would guess that your name is Milo. Am I correct?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
